# Fitness Coach Capstone - Complete Pipeline for Google Colab

This notebook executes the complete capstone pipeline with optional ablation study:
- **Step 0**: Setup and environment configuration
- **Step 1**: Mount Google Drive and clone repository
- **Step 2**: Pose backend comparison (optional)
- **Step 3-4**: Process Riccio videos → extract angles, keypoints, and labels (NPZ format)
- **Step 5**: Train BiLSTM model on extracted angles for exercise classification
- **Step 6** (Optional): Compare BiLSTM training on **normalized vs non-normalized keypoints** to validate preprocessing impact

**Total Runtime**: ~2-4 hours depending on dataset size, GPU availability, and whether comparison is enabled.

**Key Features:**
- ✅ Full Riccio dataset integration from Kaggle
- ✅ Configurable preprocessing techniques (normalization, imputation, FPS sync, rich preprocessing)
- ✅ **Normalization ablation study** (Step 6) to validate keypoint preprocessing
- ✅ GPU-optimized BiLSTM training
- ✅ Detailed comparison reports and metrics

## Step 0: Environment Setup for Colab

In [ ]:
# Check if running in Colab
import sys
try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("⚠ Not running in Colab - some features may not work as expected")

# Check GPU availability
import torch
print(f"\n GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU Name: {torch.cuda.get_device_name(0)}")
    print(f" GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 1: Mount Google Drive and Clone Repository

In [ ]:
import os
from pathlib import Path

if IN_COLAB:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted at /content/drive")
    
    # Define workspace path
    WORKSPACE_ROOT = Path("/content/drive/My Drive/Fitness-Coach-Capstone")
else:
    # Local development
    WORKSPACE_ROOT = Path.cwd()

print(f"\n Workspace root: {WORKSPACE_ROOT}")
print(f" Exists: {WORKSPACE_ROOT.exists()}")

In [ ]:
# Install dependencies if needed
import subprocess

os.chdir(WORKSPACE_ROOT)

# Install requirements
print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Optional: Install kagglehub for automatic dataset download
try:
    import kagglehub
    print("✓ kagglehub already installed")
except ImportError:
    print("Installing kagglehub for automatic dataset download...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)

print("\n✓ Dependencies installed successfully")

## Step 2: Pose Backend Comparison (Optional - Skip for Speed)

This step compares different pose detectors (MediaPipe, OpenPose, etc.). You can skip this and go directly to Step 3-4 for faster pipeline execution.

In [ ]:
# Configuration for Step 2 (Pose Comparison)
SKIP_POSE_COMPARISON = True  # Set to False to run pose comparison
NUM_VIDEOS_COMPARISON = 5    # Number of videos to analyze
OUTPUT_DIR_COMPARISON = WORKSPACE_ROOT / "results/02_pose_comparison"

if not SKIP_POSE_COMPARISON:
    print(f"Running pose backend comparison on {NUM_VIDEOS_COMPARISON} videos...")
    from fitness_coach.pipelines.run_full_comparison import main as compare_main
    
    # Prepare arguments
    comparison_args = [
        "--num-videos", str(NUM_VIDEOS_COMPARISON),
        "--output-dir", str(OUTPUT_DIR_COMPARISON),
    ]
    
    # Run comparison
    sys.argv = ["run_full_comparison.py"] + comparison_args
    try:
        compare_main()
        print("✓ Pose comparison completed")
    except Exception as e:
        print(f"✗ Pose comparison failed: {e}")
else:
    print("⊘ Pose comparison skipped (SKIP_POSE_COMPARISON=True)")

## Step 3-4: Process Riccio Videos → Extract Angles, Keypoints, and Labels

This step:
1. Downloads the Riccio dataset from Kaggle (if not already cached)
2. Extracts MediaPipe keypoints from each frame
3. Applies **full preprocessing** (torso normalization, spatial + temporal imputation, FPS sync to 30 Hz) plus **`--rich-preprocess`**: graph **Laplacian** spatial smoothing, **bone proportion**, **DWT**, and **Savitzky–Golay** temporal smoothing
4. Computes biomechanical angles and features
5. Generates NPZ files for model training

**Colab defaults below:** `MAX_VIDEOS = 400` for broader class coverage (set to `0` for the full dataset, or a small number for quick tests).

**Speed (MediaPipe is CPU-bound on Colab; GPU does not accelerate this step):** enable `MEDIAPIPE_FAST` for a lighter BlazePose model, no landmark smoothing, and 480px-long-edge detection. Use `SKIP_KEYPOINTS = True` if you only need BiLSTM angles (skips large `*_keypoints.npz`). Use `DETECTION_STRIDE = 2` to run pose every other frame (~2× fewer inferences; preprocessing still resamples to `TARGET_FPS`). If the runtime crashes or stutters, lower `RICCIO_MP_MAX_WORKERS` (e.g. `4`) or set `WORKERS = 2`.

In [ ]:
# Configuration for Steps 3-4 (Video Processing)
import os

# Output paths
RICCIO_OUTPUT_DIR = WORKSPACE_ROOT / "results/riccio_realtime_exercise_recognition"
RICCIO_STEM = "riccio_realtime_exercise_recognition"

# Processing parameters
MAX_VIDEOS = 400  # More diversity on Colab; use 0 for all videos, or e.g. 10 for a quick smoke test
WORKERS = 0      # 0 = auto-detect CPU count; use 2–4 if Colab runs out of RAM with many workers
TARGET_FPS = 30  # Resample video to this FPS

# MediaPipe / pose extraction (see riccio_kaggle_video_pipeline --help)
MEDIAPIPE_FAST = True   # True: --mediapipe-fast (model 0, no smooth, max long edge 480). False: heavier default model.
DETECTION_STRIDE = 1    # 1 = every frame; 2 = every 2nd frame (~2× faster pose, fewer input frames to FPS sync)
DETECTION_MAX_LONG_EDGE = 0  # 0 = only use pipeline default; if MEDIAPIPE_FAST, CLI sets 480. Or set e.g. 416 here and add flag below.

# Preprocessing: True = torso norm + impute + FPS + Laplacian + bone proportion + DWT + Savitzky–Golay (--rich-preprocess)
USE_RICH_PREPROCESSING = True

# Preprocessing flags
SKIP_KEYPOINTS = False          # True = BiLSTM-only path, much faster I/O and disk
DOWNLOAD_DATASET = True         # Set to True to auto-download from Kaggle

# Create output directory
RICCIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set environment variables
os.environ["RICCIO_OUTPUT_DIR"] = str(RICCIO_OUTPUT_DIR)
os.environ["RICCIO_STEM"] = RICCIO_STEM
os.environ["RICCIO_MP_MAX_WORKERS"] = "8"  # Lower to 4 if parallel workers OOM

print(f"Output directory: {RICCIO_OUTPUT_DIR}")
print(f"Output stem: {RICCIO_STEM}")
print(f"Max videos: {MAX_VIDEOS if MAX_VIDEOS > 0 else 'All'}")
print(f"Preprocessing: {'Rich (full stack)' if USE_RICH_PREPROCESSING else 'Standard (norm, impute, FPS)'}")
print(f"MediaPipe fast: {MEDIAPIPE_FAST}, detection stride: {DETECTION_STRIDE}, skip keypoints: {SKIP_KEYPOINTS}")

In [ ]:
# Run video processing pipeline (Steps 3-4)
print("\n" + "="*70)
print("STEP 3-4: Processing Riccio Videos → NPZ Format")
print("="*70 + "\n")

from fitness_coach.pipelines.riccio_kaggle_video_pipeline import main as video_pipeline_main

# Build arguments for video pipeline
video_args = [
    "--output-dir", str(RICCIO_OUTPUT_DIR),
    "--output-stem", RICCIO_STEM,
    "--target-fps", str(TARGET_FPS),
    "--workers", str(WORKERS),
]

# Add optional flags
if MAX_VIDEOS > 0:
    video_args.extend(["--max-videos", str(MAX_VIDEOS)])

if DOWNLOAD_DATASET:
    video_args.append("--download")

if SKIP_KEYPOINTS:
    video_args.append("--skip-keypoints")

if USE_RICH_PREPROCESSING:
    video_args.append("--rich-preprocess")

if MEDIAPIPE_FAST:
    video_args.append("--mediapipe-fast")
else:
    video_args.extend(["--mediapipe-model-complexity", "1"])

if int(DETECTION_STRIDE) > 1:
    video_args.extend(["--detection-stride", str(int(DETECTION_STRIDE))])

if int(DETECTION_MAX_LONG_EDGE) > 0:
    video_args.extend(["--detection-max-long-edge", str(int(DETECTION_MAX_LONG_EDGE))])

print(f"Arguments: {' '.join(video_args)}\n")

# Run pipeline
sys.argv = ["riccio_kaggle_video_pipeline.py"] + video_args
try:
    video_pipeline_main()
    print("\n✓ Video processing completed successfully")
except Exception as e:
    print(f"\n✗ Video processing failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Verify output files
import glob

print("\nVerifying output files...\n")

npz_files = {
    "Biomechanics": glob.glob(str(RICCIO_OUTPUT_DIR / f"*_biomechanics.npz")),
    "Labels": glob.glob(str(RICCIO_OUTPUT_DIR / f"*_labels.npz")),
    "Keypoints": glob.glob(str(RICCIO_OUTPUT_DIR / f"*_keypoints.npz")),
}

for file_type, files in npz_files.items():
    if files:
        print(f"✓ {file_type}: {len(files)} file(s)")
        for f in files[:3]:  # Show first 3
            size_mb = Path(f).stat().st_size / (1024**2)
            print(f"  - {Path(f).name} ({size_mb:.1f} MB)")
    else:
        print(f"✗ {file_type}: No files found")

## Step 5: Train BiLSTM (standardized vs non-standardized **angle windows**)

Uses the **same** Riccio `*_biomechanics.npz` from Steps 3–4 and runs **two** trainings:

1. **With `--standardize`** — per-feature mean/std (StandardScaler) fit on **train windows** only (Riccio §3.3.1 style).
2. **Without `--standardize`** — raw angle values fed to the network.

This is **training-time** scaling, not re-running MediaPipe. (Step 6 below is a separate ablation: **raw vs preprocessed keypoints** in the video pipeline.)

In [ ]:
# Configuration for Step 5 (BiLSTM — two runs: standardized vs not)

from pathlib import Path

# If you run this cell without Step 3–4 config, recover defaults
if "WORKSPACE_ROOT" not in dir():
    WORKSPACE_ROOT = Path.cwd()
if "RICCIO_OUTPUT_DIR" not in dir():
    RICCIO_OUTPUT_DIR = WORKSPACE_ROOT / "results/riccio_realtime_exercise_recognition"
if "RICCIO_STEM" not in dir():
    RICCIO_STEM = "riccio_realtime_exercise_recognition"

BILSTM_OUTPUT_STANDARDIZED = WORKSPACE_ROOT / "results/exercise_bilstm_standardized"
BILSTM_OUTPUT_NO_STANDARDIZE = WORKSPACE_ROOT / "results/exercise_bilstm_no_standardize"
# Alias for Step 6 / summary cells that expect BILSTM_OUTPUT_DIR
BILSTM_OUTPUT_DIR = BILSTM_OUTPUT_STANDARDIZED

BILSTM_EPOCHS = 30
BILSTM_BATCH_SIZE = 32
BILSTM_LR = 0.001
BILSTM_LEARNING_RATE = BILSTM_LR  # Step 6 cells still reference this name
STANDARDIZE_FEATURES = True  # Step 6: add --standardize when training on alternate NPZs

BILSTM_ARCHITECTURE = "cnn_attn"
CLASSIFICATION_ONLY = True
EVAL_ON_TEST = True
VAL_RATIO = 0.15
TEST_RATIO = 0.15

for d in (BILSTM_OUTPUT_STANDARDIZED, BILSTM_OUTPUT_NO_STANDARDIZE):
    d.mkdir(parents=True, exist_ok=True)

print("NPZ dir:", RICCIO_OUTPUT_DIR)
print("Out (standardized windows):", BILSTM_OUTPUT_STANDARDIZED)
print("Out (no window standardize):", BILSTM_OUTPUT_NO_STANDARDIZE)
print(f"Epochs={BILSTM_EPOCHS} batch={BILSTM_BATCH_SIZE} lr={BILSTM_LR} arch={BILSTM_ARCHITECTURE}")

In [ ]:
# Run BiLSTM twice: (1) standardized angle windows (2) raw angle windows
print("\n" + "="*70)
print("STEP 5: BiLSTM — standardized vs non-standardized inputs")
print("="*70 + "\n")

bio = RICCIO_OUTPUT_DIR / f"{RICCIO_STEM}_biomechanics.npz"
lab = RICCIO_OUTPUT_DIR / f"{RICCIO_STEM}_labels.npz"
assert bio.is_file(), f"Missing {bio} — run Steps 3–4 first"
assert lab.is_file(), f"Missing {lab} — run Steps 3–4 first"

from fitness_coach.training.train_exercise_bilstm import main as train_bilstm_main


def run_bilstm(output_dir: Path, standardize: bool, label: str) -> None:
    args = [
        "--preset", "riccio",
        "--kaggle-angles-dir", str(RICCIO_OUTPUT_DIR),
        "--kaggle-stem", RICCIO_STEM,
        "--output-dir", str(output_dir),
        "--epochs", str(BILSTM_EPOCHS),
        "--batch-size", str(BILSTM_BATCH_SIZE),
        "--lr", str(BILSTM_LR),
        "--architecture", BILSTM_ARCHITECTURE,
        "--kaggle-val-ratio", str(VAL_RATIO),
        "--kaggle-test-ratio", str(TEST_RATIO),
    ]
    if standardize:
        args.append("--standardize")
    if CLASSIFICATION_ONLY:
        args.append("--classification-only")
    if EVAL_ON_TEST:
        args.append("--eval-test")
    print(f"\n>>> {label}\n    python train_exercise_bilstm.py {' '.join(args)}\n")
    sys.argv = ["train_exercise_bilstm.py"] + args
    train_bilstm_main()


try:
    run_bilstm(BILSTM_OUTPUT_STANDARDIZED, True, "A: StandardScaler on train windows (--standardize)")
    run_bilstm(BILSTM_OUTPUT_NO_STANDARDIZE, False, "B: No window scaling (omit --standardize)")
    print("\n✓ Step 5 complete — checkpoints in exercise_bilstm_standardized / exercise_bilstm_no_standardize")
except Exception as e:
    print(f"\n✗ Step 5 failed: {e}")
    import traceback
    traceback.print_exc()

## Step 6: Compare BiLSTM Training on Normalized vs Non-Normalized Keypoints

This step compares the impact of keypoint normalization on model performance by:
1. Processing the same videos **without normalization** (raw keypoints from MediaPipe)
2. Training a BiLSTM model on the non-normalized angles
3. Comparing test accuracy, F1-scores, and other metrics between both versions
4. Generating a detailed comparison report

This ablation study helps validate whether normalization improves exercise classification robustness.

In [ ]:
%%time
# Configuration for Step 6 (Normalization Comparison)

# Enable/disable comparison
RUN_NORMALIZATION_COMPARISON = True  # Set to False to skip this step

if not RUN_NORMALIZATION_COMPARISON:
    print("⊘ Normalization comparison skipped (RUN_NORMALIZATION_COMPARISON=False)")
else:
    # Output paths for non-normalized version
    RICCIO_OUTPUT_DIR_NO_NORM = WORKSPACE_ROOT / "results/riccio_realtime_exercise_recognition_no_norm"
    RICCIO_STEM_NO_NORM = "riccio_realtime_exercise_recognition_no_norm"
    
    # Training output paths
    BILSTM_OUTPUT_DIR_NORMALIZED = BILSTM_OUTPUT_DIR
    BILSTM_OUTPUT_DIR_NO_NORM = WORKSPACE_ROOT / "results/exercise_bilstm_no_norm"
    COMPARISON_OUTPUT_DIR = WORKSPACE_ROOT / "results/normalization_comparison"
    
    # Create output directories
    RICCIO_OUTPUT_DIR_NO_NORM.mkdir(parents=True, exist_ok=True)
    BILSTM_OUTPUT_DIR_NO_NORM.mkdir(parents=True, exist_ok=True)
    COMPARISON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    print(f"Normalized version output: {BILSTM_OUTPUT_DIR_NORMALIZED}")
    print(f"Non-normalized version output: {BILSTM_OUTPUT_DIR_NO_NORM}")
    print(f"Comparison report output: {COMPARISON_OUTPUT_DIR}")

In [ ]:
%%time
# Step 6a: Process Riccio videos WITHOUT normalization
if RUN_NORMALIZATION_COMPARISON:
    print("\n" + "="*70)
    print("STEP 6a: Processing Riccio Videos → NPZ (Raw Keypoints, No Normalization)")
    print("="*70 + "\n")
    
    # Build arguments for video pipeline WITHOUT normalization
    video_args_no_norm = [
        "--output-dir", str(RICCIO_OUTPUT_DIR_NO_NORM),
        "--output-stem", RICCIO_STEM_NO_NORM,
        "--target-fps", str(TARGET_FPS),
        "--workers", str(WORKERS),
        # MediaPipe xy without torso norm / impute / FPS stack (see --raw-keypoints in pipeline)
        "--raw-keypoints",
    ]
    
    # Add optional flags (keep consistent with normalized version)
    if MAX_VIDEOS > 0:
        video_args_no_norm.extend(["--max-videos", str(MAX_VIDEOS)])
    
    if DOWNLOAD_DATASET:
        video_args_no_norm.append("--download")
    
    if SKIP_KEYPOINTS:
        video_args_no_norm.append("--skip-keypoints")
    
    # DO NOT use rich preprocessing for raw keypoints comparison
    if globals().get("MEDIAPIPE_FAST", True):
        video_args_no_norm.append("--mediapipe-fast")
    _ds = int(globals().get("DETECTION_STRIDE", 1))
    if _ds > 1:
        video_args_no_norm.extend(["--detection-stride", str(_ds)])
    _mle = int(globals().get("DETECTION_MAX_LONG_EDGE", 0))
    if _mle > 0:
        video_args_no_norm.extend(["--detection-max-long-edge", str(_mle)])
    
    print(f"Arguments: {' '.join(video_args_no_norm)}\n")

    from fitness_coach.pipelines.riccio_kaggle_video_pipeline import main as video_pipeline_main

    # Run pipeline
    sys.argv = ["riccio_kaggle_video_pipeline.py"] + video_args_no_norm
    try:
        video_pipeline_main()
        print("\n✓ Non-normalized video processing completed successfully")
    except Exception as e:
        print(f"\n✗ Non-normalized video processing failed: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
%%time
# Step 6b: Train BiLSTM on non-normalized keypoints
if RUN_NORMALIZATION_COMPARISON:
    print("\n" + "="*70)
    print("STEP 6b: Training BiLSTM on Non-Normalized Keypoints")
    print("="*70 + "\n")
    
    # Verify non-normalized output files exist
    print("Verifying non-normalized output files...")
    npz_files_no_norm = {
        "Biomechanics": list(RICCIO_OUTPUT_DIR_NO_NORM.glob("*_biomechanics.npz")),
        "Labels": list(RICCIO_OUTPUT_DIR_NO_NORM.glob("*_labels.npz")),
    }
    
    all_files_exist = all(len(files) > 0 for files in npz_files_no_norm.values())
    if not all_files_exist:
        print("✗ Non-normalized files not found. Skipping BiLSTM training on non-normalized data.\n")
    else:
        print("✓ Non-normalized output files verified\n")
        
        # Build arguments for BiLSTM training on non-normalized data
        train_args_no_norm = [
            "--preset", "riccio",
            "--kaggle-angles-dir", str(RICCIO_OUTPUT_DIR_NO_NORM),
            "--kaggle-stem", RICCIO_STEM_NO_NORM,
            "--output-dir", str(BILSTM_OUTPUT_DIR_NO_NORM),
            "--epochs", str(BILSTM_EPOCHS),
            "--batch-size", str(BILSTM_BATCH_SIZE),
            "--lr", str(BILSTM_LEARNING_RATE),
            "--architecture", BILSTM_ARCHITECTURE,
            "--kaggle-val-ratio", str(VAL_RATIO),
            "--kaggle-test-ratio", str(TEST_RATIO),
            "--kaggle-seed", "42",  # Use same seed for fair comparison
        ]
        
        # Add optional flags
        if STANDARDIZE_FEATURES:
            train_args_no_norm.append("--standardize")
        
        if CLASSIFICATION_ONLY:
            train_args_no_norm.append("--classification-only")
        
        if EVAL_ON_TEST:
            train_args_no_norm.append("--eval-test")
        
        print(f"Arguments: {' '.join(train_args_no_norm)}\n")

        from fitness_coach.training.train_exercise_bilstm import main as train_bilstm_main

        # Run training
        sys.argv = ["train_exercise_bilstm.py"] + train_args_no_norm
        try:
            train_bilstm_main()
            print("\n✓ BiLSTM training on non-normalized keypoints completed successfully")
        except Exception as e:
            print(f"\n✗ BiLSTM training on non-normalized keypoints failed: {e}")
            import traceback
            traceback.print_exc()

In [ ]:
%%time
# Step 6c: Compare results (Normalized vs Non-Normalized)
if RUN_NORMALIZATION_COMPARISON:
    print("\n" + "="*70)
    print("STEP 6c: Comparison Analysis - Normalized vs Non-Normalized Keypoints")
    print("="*70 + "\n")
    
    import json
    import pandas as pd
    
    # Load metrics from both training runs
    metrics_normalized = None
    metrics_no_norm = None
    
    # Load normalized metrics
    normalized_metrics_file = BILSTM_OUTPUT_DIR_NORMALIZED / "test_classification_metrics.json"
    if normalized_metrics_file.exists():
        with open(normalized_metrics_file) as f:
            metrics_normalized = json.load(f)
        print(f"✓ Loaded normalized training metrics")
    else:
        print(f"✗ Normalized metrics not found: {normalized_metrics_file}")
    
    # Load non-normalized metrics
    no_norm_metrics_file = BILSTM_OUTPUT_DIR_NO_NORM / "test_classification_metrics.json"
    if no_norm_metrics_file.exists():
        with open(no_norm_metrics_file) as f:
            metrics_no_norm = json.load(f)
        print(f"✓ Loaded non-normalized training metrics")
    else:
        print(f"✗ Non-normalized metrics not found: {no_norm_metrics_file}")
    
    print()
    
    # Compare metrics if both exist
    if metrics_normalized and metrics_no_norm:
        print("\n📊 COMPARISON RESULTS\n")
        print("-" * 70)
        
        # Extract key metrics
        comparison_data = {
            "Metric": [],
            "Normalized": [],
            "Non-Normalized": [],
            "Delta": [],
            "Better": []
        }
        
        key_metrics = ["accuracy", "f1_macro", "f1_weighted", "recall", "precision"]
        
        for metric in key_metrics:
            norm_val = metrics_normalized.get(metric)
            no_norm_val = metrics_no_norm.get(metric)
            
            if norm_val is not None and no_norm_val is not None:
                delta = no_norm_val - norm_val
                better = "Normalized" if delta < 0 else "Non-Normalized" if delta > 0 else "Tie"
                
                comparison_data["Metric"].append(metric.replace("_", " ").title())
                comparison_data["Normalized"].append(f"{norm_val:.4f}")
                comparison_data["Non-Normalized"].append(f"{no_norm_val:.4f}")
                comparison_data["Delta"].append(f"{delta:+.4f}")
                comparison_data["Better"].append(better)
        
        # Create comparison DataFrame
        df_comparison = pd.DataFrame(comparison_data)
        print(df_comparison.to_string(index=False))
        
        print("\n" + "-" * 70)
        print("\nKey Findings:")
        print("-" * 70)
        
        # Summary statistics
        norm_acc = metrics_normalized.get("accuracy", 0)
        no_norm_acc = metrics_no_norm.get("accuracy", 0)
        acc_delta = no_norm_acc - norm_acc
        
        print(f"\n• Normalized Accuracy:     {norm_acc:.4f}")
        print(f"• Non-Normalized Accuracy: {no_norm_acc:.4f}")
        print(f"• Difference:              {acc_delta:+.4f} ({acc_delta*100:+.2f}%)\n")
        
        if abs(acc_delta) < 0.01:
            print("  → Normalization has minimal impact on accuracy")
        elif acc_delta > 0:
            print("  → Non-normalized keypoints perform BETTER")
            print("     (Torso normalization may remove useful camera-distance information)")
        else:
            print("  → Normalized keypoints perform BETTER")
            print("     (Torso normalization improves robustness to camera distance variations)")
        
        # Save comparison report
        report_path = COMPARISON_OUTPUT_DIR / "normalization_comparison.json"
        report = {
            "timestamp": str(pd.Timestamp.now()),
            "preprocessing_config": {
                "normalized": {
                    "strategy": "standard (normalization + imputation + FPS sync)",
                    "output_dir": str(BILSTM_OUTPUT_DIR_NORMALIZED),
                },
                "non_normalized": {
                    "strategy": "raw keypoints (minimal preprocessing)",
                    "output_dir": str(BILSTM_OUTPUT_DIR_NO_NORM),
                }
            },
            "training_config": {
                "architecture": BILSTM_ARCHITECTURE,
                "epochs": BILSTM_EPOCHS,
                "batch_size": BILSTM_BATCH_SIZE,
                "learning_rate": BILSTM_LEARNING_RATE,
            },
            "comparison_results": {
                "normalized": metrics_normalized,
                "non_normalized": metrics_no_norm,
                "deltas": {
                    "accuracy_delta": acc_delta,
                    f1_macro_delta": metrics_no_norm.get("f1_macro", 0) - metrics_normalized.get("f1_macro", 0),
                }
            }
        }
        
        with open(report_path, "w") as f:
            json.dump(report, f, indent=2)
        
        print(f"\n✓ Comparison report saved: {report_path}")
    else:
        print("⚠ Cannot generate comparison: missing metrics from one or both training runs")

## Summary: Results and Artifacts

In [ ]:
# Display summary of generated artifacts
import json
from pathlib import Path

print("\n" + "="*70)
print("PIPELINE SUMMARY")
print("="*70 + "\n")

# Check preprocessing outputs
print("📊 PREPROCESSING OUTPUTS (Step 3-4)")
print("-" * 70)
preprocess_dir = RICCIO_OUTPUT_DIR
if preprocess_dir.exists():
    npz_files = list(preprocess_dir.glob("*.npz"))
    print(f"Generated {len(npz_files)} NPZ files (Normalized):")
    total_size = 0
    for f in sorted(npz_files):
        size_mb = f.stat().st_size / (1024**2)
        total_size += size_mb
        print(f"  • {f.name}: {size_mb:.1f} MB")
    print(f"\nTotal preprocessing data (normalized): {total_size:.1f} MB")
else:
    print(f"Directory not found: {preprocess_dir}")

# Check non-normalized preprocessing (if comparison was run)
if RUN_NORMALIZATION_COMPARISON:
    preprocess_dir_no_norm = RICCIO_OUTPUT_DIR_NO_NORM
    if preprocess_dir_no_norm.exists():
        npz_files_no_norm = list(preprocess_dir_no_norm.glob("*.npz"))
        if npz_files_no_norm:
            print(f"\nGenerated {len(npz_files_no_norm)} NPZ files (Non-Normalized):")
            total_size_no_norm = 0
            for f in sorted(npz_files_no_norm):
                size_mb = f.stat().st_size / (1024**2)
                total_size_no_norm += size_mb
                print(f"  • {f.name}: {size_mb:.1f} MB")
            print(f"\nTotal preprocessing data (non-normalized): {total_size_no_norm:.1f} MB")

print("\n")

# Check training outputs
print("🎯 TRAINING OUTPUTS (Step 5 & Step 6)")
print("-" * 70)

# Normalized training
training_dir = BILSTM_OUTPUT_DIR
if training_dir.exists():
    checkpoints = list(training_dir.glob("*.pth"))
    if checkpoints:
        print(f"\nNormalized Model Checkpoints: {len(checkpoints)}")
        for cp in sorted(checkpoints)[:3]:  # Show first 3
            size_mb = cp.stat().st_size / (1024**2)
            print(f"  • {cp.name}: {size_mb:.1f} MB")
    
    metrics_file = training_dir / "test_classification_metrics.json"
    if metrics_file.exists():
        with open(metrics_file) as f:
            metrics = json.load(f)
        print(f"\nNormalized Training Metrics:")
        if "accuracy" in metrics:
            print(f"  • Test Accuracy: {metrics['accuracy']:.4f}")
        if "f1_macro" in metrics:
            print(f"  • F1 (Macro): {metrics['f1_macro']:.4f}")

# Non-normalized training (if comparison was run)
if RUN_NORMALIZATION_COMPARISON and BILSTM_OUTPUT_DIR_NO_NORM.exists():
    checkpoints_no_norm = list(BILSTM_OUTPUT_DIR_NO_NORM.glob("*.pth"))
    if checkpoints_no_norm:
        print(f"\nNon-Normalized Model Checkpoints: {len(checkpoints_no_norm)}")
        for cp in sorted(checkpoints_no_norm)[:3]:  # Show first 3
            size_mb = cp.stat().st_size / (1024**2)
            print(f"  • {cp.name}: {size_mb:.1f} MB")
    
    metrics_file_no_norm = BILSTM_OUTPUT_DIR_NO_NORM / "test_classification_metrics.json"
    if metrics_file_no_norm.exists():
        with open(metrics_file_no_norm) as f:
            metrics_no_norm = json.load(f)
        print(f"\nNon-Normalized Training Metrics:")
        if "accuracy" in metrics_no_norm:
            print(f"  • Test Accuracy: {metrics_no_norm['accuracy']:.4f}")
        if "f1_macro" in metrics_no_norm:
            print(f"  • F1 (Macro): {metrics_no_norm['f1_macro']:.4f}")

# Comparison report
if RUN_NORMALIZATION_COMPARISON:
    comparison_report = COMPARISON_OUTPUT_DIR / "normalization_comparison.json"
    if comparison_report.exists():
        print(f"\n📈 Normalization Comparison Report:")
        print(f"  • Report: {comparison_report.name}")
        with open(comparison_report) as f:
            comp = json.load(f)
        if "comparison_results" in comp and "deltas" in comp["comparison_results"]:
            acc_delta = comp["comparison_results"]["deltas"].get("accuracy_delta", 0)
            print(f"  • Accuracy Delta: {acc_delta:+.4f}")
            if abs(acc_delta) < 0.01:
                print(f"  • Impact: Minimal (< 1%)")
            elif acc_delta > 0:
                print(f"  • Winner: Non-Normalized (+{acc_delta*100:.2f}%)")
            else:
                print(f"  • Winner: Normalized ({acc_delta*100:.2f}%)")

print("\n" + "="*70)
print("✓ Pipeline execution complete!")
print("="*70)

## Optional: Download Results from Google Drive

If running in Colab, you can download the results to Google Drive.

In [ ]:
if IN_COLAB:
    import shutil
    
    drive_results = Path("/content/drive/My Drive/Fitness-Coach-Results")
    drive_results.mkdir(exist_ok=True)
    
    # Copy results
    results_local = WORKSPACE_ROOT / "results"
    if results_local.exists():
        print(f"Copying results to Google Drive...")
        
        for source_dir in results_local.iterdir():
            if source_dir.is_dir():
                dest = drive_results / source_dir.name
                if dest.exists():
                    shutil.rmtree(dest)
                shutil.copytree(source_dir, dest)
                print(f"  ✓ Copied {source_dir.name}")
        
        print(f"\nResults available at: {drive_results}")
else:
    print("Not running in Colab - results are already in the workspace.")
    print(f"Results location: {WORKSPACE_ROOT / 'results'}")